In [1]:
import pandas as pd

ir2_test = pd.read_csv('/apps/data/AHS_project/Data/ir2_background.csv')
df_counts = ir2_test['ID'].value_counts().reset_index()
df_counts.columns = ['ID', 'count']
print(df_counts)
print(df_counts['count'].value_counts())

         ID  count
0    329214     29
1    339734     26
2    262366     25
3    331148     24
4    105054     23
..      ...    ...
316  232005      1
317  231949      1
318  231242      1
319  229187      1
320  398129      1

[321 rows x 2 columns]
1     89
2     69
3     37
4     20
5     19
6     14
8     10
7      9
11     8
9      5
12     5
13     4
14     4
21     4
17     3
23     3
15     2
16     2
10     2
18     2
19     2
20     2
22     2
26     1
24     1
25     1
29     1
Name: count, dtype: int64


In [2]:
predictions = pd.read_csv('/apps/data/AHS_project/met_label_x_predictions_ensmeble_max_04.csv')
predictions = pd.merge(predictions, ir2_test, on='File Name', how='inner')
report_counts = predictions.groupby('ID').size().reset_index(name='num_reports')
# Merge back into the original DataFrame
predictions = predictions.merge(report_counts, on='ID')
bins = [0, 1, 2, 4, 8, 14, 23, float('inf')]
labels = ["1", "2", "3-4", "5-8", "9–14", "15-23", ">=24"]

predictions['bin'] = pd.cut(predictions['num_reports'], bins=bins, labels=labels, right=True)
print(predictions.columns)
print(predictions.shape)

Index(['Unnamed: 0', 'File Name', 'Labels', 'Predicted', 'Probabilities',
       'Findings_prob', 'Impression_prob', 'ID', 'Metastasis', 'Notes',
       'Has_History', 'met_label_x', 'Valerie_Note', 'bin',
       'met_label_x_weight', 'full_report', 'findings', 'impressions',
       'background', 'impressions_back', 'sections', 'findings_tokens',
       'impressions_tokens', 'num_reports'],
      dtype='object')
(1585, 24)


In [3]:
reports_per_bin = predictions[predictions['Predicted'] == 1].groupby('bin').size()
print(reports_per_bin)
reports_per_bin_label = predictions[predictions['Labels'] == 1].groupby('bin').size()
print(reports_per_bin_label)

bin
1          3
2         10
3-4       20
5-8       27
9–14      63
15-23    158
>=24      51
dtype: int64
bin
1          1
2          6
3-4       14
5-8       23
9–14      44
15-23    135
>=24      53
dtype: int64


In [4]:
patients_per_bin = (
    predictions[predictions['Predicted'] == 1]
    .groupby('bin')['ID']
    .nunique()
)
print(patients_per_bin)


patients_per_bin_label = (
    predictions[predictions['Labels'] == 1]
    .groupby('bin')['ID']
    .nunique()
)
print(patients_per_bin_label)

bin
1         3
2         8
3-4      11
5-8       9
9–14     15
15-23    17
>=24      3
Name: ID, dtype: int64
bin
1        1
2        4
3-4      5
5-8      4
9–14     5
15-23    8
>=24     2
Name: ID, dtype: int64


In [5]:
ir2_test = pd.read_csv('/apps/data/AHS_project/Data/ir2_background_not_none.csv')
hanxiao_data = pd.read_csv('/apps/data/AHS_project/Data/ir_analysis/ir2_nonmelanoma_prediag_cns_excluded_file_present.csv')

new_ir2 = ir2_test[ir2_test['File Name'].isin(hanxiao_data['New File Name'])]

print(new_ir2.shape)
print(new_ir2.columns)
new_ir2.to_csv('/apps/data/AHS_project/Data/ir_analysis/ir2_28_cohort.csv', index=False)
new_ir2.to_pickle('/apps/data/AHS_project/Data/ir_analysis/ir2_28_cohort.pkl')

(1230, 17)
Index(['ID', 'File Name', 'Metastasis', 'Notes', 'Has_History', 'met_label_x',
       'Valerie_Note', 'bin', 'met_label_x_weight', 'full_report', 'findings',
       'impressions', 'background', 'impressions_back', 'sections',
       'findings_tokens', 'impressions_tokens'],
      dtype='object')


In [7]:
import pandas as pd
import numpy as np

# ---------- 0) Load ----------
ir2_all  = pd.read_csv('/apps/data/AHS_project/Resources/ir2_weight_by_patient.csv')[['ID', 'File Name']]
ir2_test = pd.read_csv('/apps/data/AHS_project/Data/ir2_background_not_none.csv')[['ID', 'File Name', 'Metastasis']]

# the file name is new file name here
ir2_test = ir2_test[['ID', 'File Name', 'Metastasis']]
print(ir2_all.columns)
print(ir2_test.columns)
# If you also need prediction/label analysis on ir2_test rows:
predictions = pd.read_csv('/apps/data/AHS_project/met_label_x_ir2_predictions_ensmeble_max_04.csv')
predictions['Impression_pred'] = np.where(predictions['Impression_prob'] >= 0.4, 1, 0)
predictions['Findings_pred'] = np.where(predictions['Findings_prob'] >= 0.4, 1, 0)

# ---------- 1) Build per-ID bin labels from ir2_all ----------
# Per-ID number of reports in ir2_all
counts_all = ir2_all.groupby('ID').size().reset_index(name='num_reports')

# Define bins & labels (right-closed: …,1], …, 23], (23, inf))
bins   = [0, 1, 2, 4, 8, 14, 23, np.inf]
labels = ["1", "2", "3-4", "5-8", "9–14", "15-23", ">=24"]

counts_all['bin'] = pd.cut(
    counts_all['num_reports'],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)

# Attach bin label back to every row in ir2_all
ir2_all = ir2_all.merge(counts_all[['ID', 'num_reports', 'bin']], on='ID', how='left')

# ---------- 2) Propagate those bin labels to ir2_test ----------
ir2_test = ir2_test.merge(counts_all[['ID', 'num_reports', 'bin']], on='ID', how='left')
print(ir2_test.columns)
# Now ir2_test has columns: …, ID, num_reports (from ir2_all), bin

# ---------- 3) Analyses on ir2_test using the bin label ----------

# 3a) Total reports per bin (how many rows of ir2_test per bin)
reports_per_bin = ir2_test.groupby('bin').size()
print("ir2_test — Reports per bin:")
print(reports_per_bin.reindex(labels).fillna(0).astype(int))

# 3b) Unique patients per bin in ir2_test
patients_per_bin = ir2_test.groupby('bin')['ID'].nunique()
print("\nir2_test — Unique patients per bin:")
print(patients_per_bin.reindex(labels).fillna(0).astype(int))

predictions['Predicted'] = predictions['Impression_pred']
# predictions['Labels'] = predictions['labels']
predictions = predictions.merge(ir2_test, on='File Name', how='inner')
print(predictions.columns)
# Reports per bin where Predicted == 1
reports_pred1 = predictions.loc[predictions['Predicted'] == 1].groupby('bin').size()
print("\nPredicted==1 — Reports per bin:")
print(reports_pred1.reindex(labels).fillna(0).astype(int))

# Reports per bin where Labels == 1
# reports_label1 = predictions.loc[predictions['Metastasis'].astype(str).str.contains('Yes') | predictions['Labels'] == 1].groupby('bin').size()
reports_label1 = predictions.loc[predictions['Metastasis'].astype(str).str.contains('Yes')].groupby('bin').size()

print("\nLabels==1 — Reports per bin:")
print(reports_label1.reindex(labels).fillna(0).astype(int))

# Unique patients per bin (Predicted == 1)
patients_pred1 = (predictions.loc[predictions['Predicted'] == 1]
                  .groupby('bin')['ID'].nunique())
print("\nPredicted==1 — Unique patients per bin:")
print(patients_pred1.reindex(labels).fillna(0).astype(int))

# Unique patients per bin (Labels == 1)
# patients_label1 = (predictions.loc[predictions['Labels'] == 1]
#                    .groupby('bin')['ID'].nunique())
patients_label1 = (
    predictions.loc[predictions['Metastasis'].astype(str).str.contains('Yes')]
    .groupby('bin')['ID'].nunique()
)
print("\nLabels==1 — Unique patients per bin:")
print(patients_label1.reindex(labels).fillna(0).astype(int))


# Build a TP mask (both predicted 1 and label 1)
tp = ((predictions['Metastasis'].astype(str).str.contains('Yes'))) & (predictions['Predicted'] == 1)

print('total met yes:', len(predictions[predictions['Metastasis'].astype(str).str.contains('Yes')]))
# print('label total met yes:', len(predictions.loc[predictions['Labels'] == 1]))
# Reports per bin (TP)
reports_tp = (
    predictions.loc[tp]
    .groupby('bin', dropna=False)
    .size()
    .reindex(labels)              # ensure all bins print
    .fillna(0).astype(int)
)
print("\nTP — Reports per bin:")
print(reports_tp)

# Unique patients per bin (TP)
patients_tp = (
    predictions.loc[tp]
    .groupby('bin', dropna=False)['ID']
    .nunique()
    .reindex(labels)
    .fillna(0).astype(int)
)
print("\nTP — Unique patients per bin:")
print(patients_tp)

Index(['ID', 'File Name'], dtype='object')
Index(['ID', 'File Name', 'Metastasis'], dtype='object')
Index(['ID', 'File Name', 'Metastasis', 'num_reports', 'bin'], dtype='object')
ir2_test — Reports per bin:
bin
1         97
2        154
3-4      193
5-8      293
9–14     429
15-23    431
>=24     236
dtype: int64

ir2_test — Unique patients per bin:
bin
1        97
2        79
3-4      60
5-8      50
9–14     40
15-23    22
>=24      9
Name: ID, dtype: int64
Index(['Unnamed: 0', 'File Name', 'Labels', 'Predicted', 'Probabilities',
       'Findings_prob', 'Impression_prob', 'Impression_pred', 'Findings_pred',
       'ID', 'Metastasis', 'num_reports', 'bin'],
      dtype='object')

Predicted==1 — Reports per bin:
bin
1          1
2         10
3-4       11
5-8       25
9–14      61
15-23    123
>=24      62
dtype: int64

Labels==1 — Reports per bin:
bin
1          1
2          7
3-4        9
5-8       21
9–14      57
15-23    144
>=24      80
dtype: int64

Predicted==1 — Unique patients p

In [8]:
import pandas as pd
import numpy as np

# ---------- 0) Load ----------
ir2_all  = pd.read_csv('/apps/data/AHS_project/Resources/ir2_weight_by_patient.csv')[['ID', 'File Name']]
ir2_test = pd.read_csv('/apps/data/AHS_project/Data/ir2_background_not_none.csv')[['ID', 'File Name', 'Metastasis']]
print(ir2_all.columns)
print(ir2_test.columns)
# If you also need prediction/label analysis on ir2_test rows:
# predictions = pd.read_csv('/apps/data/AHS_project/met_label_x_ir2_predictions_ensmeble_max_04.csv')

# ---------- 1) Build per-ID bin labels from ir2_all ----------
# Per-ID number of reports in ir2_all
counts_all = ir2_all.groupby('ID').size().reset_index(name='num_reports')

# Define bins & labels (right-closed: …,1], …, 23], (23, inf))
bins   = [0, 1, 2, 4, 8, 14, 23, np.inf]
labels = ["1", "2", "3-4", "5-8", "9–14", "15-23", ">=24"]

counts_all['bin'] = pd.cut(
    counts_all['num_reports'],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)

# Attach bin label back to every row in ir2_all
ir2_all = ir2_all.merge(counts_all[['ID', 'num_reports', 'bin']], on='ID', how='left')

# ---------- 2) Propagate those bin labels to ir2_test ----------
ir2_test = ir2_test.merge(counts_all[['ID', 'num_reports', 'bin']], on='ID', how='left')
print(ir2_test.columns)
# Now ir2_test has columns: …, ID, num_reports (from ir2_all), bin

# ---------- 3) Analyses on ir2_test using the bin label ----------

# 3a) Total reports per bin (how many rows of ir2_test per bin)
reports_per_bin = ir2_test.groupby('bin').size()
print("ir2_test — Reports per bin:")
print(reports_per_bin.reindex(labels).fillna(0).astype(int))

# 3b) Unique patients per bin in ir2_test
patients_per_bin = ir2_test.groupby('bin')['ID'].nunique()
print("\nir2_test — Unique patients per bin:")
print(patients_per_bin.reindex(labels).fillna(0).astype(int))

# ---------- 4) (Optional) If you want Predicted/Labels analysis on ir2_test ----------
# Merge predictions onto ir2_test to get ID + bin per prediction row
# (Assumes predictions has 'File Name', 'Predicted', 'Labels')
predictions = pd.read_csv('/apps/data/AHS_project/met_label_x_ir2_predictions_ensmeble_max_04.csv')
predictions['Impression_pred'] = np.where(predictions['Impression_prob'] >= 0.4, 1, 0)
predictions['Findings_pred'] = np.where(predictions['Findings_prob'] >= 0.4, 1, 0)
# predictions['Predicted'] = predictions['Findings_pred']

predictions = predictions.merge(ir2_test[['File Name', 'ID', 'bin', 'Metastasis']], on='File Name', how='inner')
print(predictions.columns)
# Reports per bin where Predicted == 1
reports_pred1 = predictions.loc[predictions['Predicted'] == 1].groupby('bin').size()
print("\nPredicted==1 — Reports per bin:")
print(reports_pred1.reindex(labels).fillna(0).astype(int))

# Reports per bin where Labels == 1
reports_label1 = predictions.loc[predictions['Metastasis'].astype(str).str.contains('Yes')].groupby('bin').size()
print("\nLabels==1 — Reports per bin:")
print(reports_label1.reindex(labels).fillna(0).astype(int))

# Unique patients per bin (Predicted == 1)
patients_pred1 = (predictions.loc[predictions['Predicted'] == 1]
                  .groupby('bin')['ID'].nunique())
print("\nPredicted==1 — Unique patients per bin:")
print(patients_pred1.reindex(labels).fillna(0).astype(int))

# Unique patients per bin (Labels == 1)
# patients_label1 = (predictions.loc[predictions['Labels'] == 1]
#                    .groupby('bin')['ID'].nunique())
patients_label1 = (
    predictions.loc[predictions['Metastasis'].astype(str).str.contains('Yes')]
    .groupby('bin')['ID'].nunique()
)
print("\nLabels==1 — Unique patients per bin:")
print(patients_label1.reindex(labels).fillna(0).astype(int))


# Build a TP mask (both predicted 1 and label 1)
tp = ((predictions['Metastasis'].astype(str).str.contains('Yes'))) & (predictions['Predicted'] == 1)

print('total met yes:', len(predictions[predictions['Metastasis'].astype(str).str.contains('Yes')]))
print('label total met yes:', len(predictions.loc[predictions['Labels'] == 1]))
# Reports per bin (TP)
reports_tp = (
    predictions.loc[tp]
    .groupby('bin', dropna=False)
    .size()
    .reindex(labels)              # ensure all bins print
    .fillna(0).astype(int)
)
print("\nTP — Reports per bin:")
print(reports_tp)

# Unique patients per bin (TP)
patients_tp = (
    predictions.loc[tp]
    .groupby('bin', dropna=False)['ID']
    .nunique()
    .reindex(labels)
    .fillna(0).astype(int)
)
print("\nTP — Unique patients per bin:")
print(patients_tp)

Index(['ID', 'File Name'], dtype='object')
Index(['ID', 'File Name', 'Metastasis'], dtype='object')
Index(['ID', 'File Name', 'Metastasis', 'num_reports', 'bin'], dtype='object')
ir2_test — Reports per bin:
bin
1         97
2        154
3-4      193
5-8      293
9–14     429
15-23    431
>=24     236
dtype: int64

ir2_test — Unique patients per bin:
bin
1        97
2        79
3-4      60
5-8      50
9–14     40
15-23    22
>=24      9
Name: ID, dtype: int64
Index(['Unnamed: 0', 'File Name', 'Labels', 'Predicted', 'Probabilities',
       'Findings_prob', 'Impression_prob', 'Impression_pred', 'Findings_pred',
       'ID', 'bin', 'Metastasis'],
      dtype='object')

Predicted==1 — Reports per bin:
bin
1          6
2         17
3-4       19
5-8       34
9–14      90
15-23    149
>=24      83
dtype: int64

Labels==1 — Reports per bin:
bin
1          1
2          7
3-4        9
5-8       21
9–14      57
15-23    144
>=24      80
dtype: int64

Predicted==1 — Unique patients per bin:
bin
1  